### Feature Engineering

#### Creating new features in our dataset for modelling

#### Importing Packages and Data

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
project_root = Path(".").resolve().parent
input_path = project_root / "data" / "processed" / "london_clean.parquet"
output_path = project_root / "data" / "processed" / "london_features.parquet"

df = pd.read_parquet(input_path)

In [3]:
df

,price,date_of_transfer,postcode,property_type,old_new,duration,district
0,215000,2025-01-31,E3 2PQ,F,N,L,TOWER HAMLETS
1,349000,2025-12-22,E17 7LB,F,N,L,WALTHAM FOREST
2,540000,2025-12-16,NW4 3PG,S,N,F,BARNET
3,460000,2025-12-15,E2 8FZ,F,N,L,HACKNEY
4,265000,2025-12-19,E14 9BF,F,N,L,TOWER HAMLETS
...,...,...,...,...,...,...,...
72587,393800,2025-06-11,E10 7PE,F,N,L,WALTHAM FOREST
72588,388000,2025-05-29,RM13 9SN,S,N,F,HAVERING
72589,358000,2025-06-05,RM3 7DX,T,N,F,HAVERING
72590,1095000,2025-01-03,E18 2PS,S,N,F,REDBRIDGE


#### Features

In [44]:
# Postcode Area: Letters at start of postcode
df["postcode_area"] = df["postcode"].str.extract(r'([A-Z]+)', expand=False)

df["postcode_area"]

0         E
1         E
2        NW
3         E
4         E
         ..
72587     E
72588    RM
72589    RM
72590     E
72591    RM
Name: postcode_area, Length: 72592, dtype: object

In [45]:
# Postcode District: Outward code (area + district)
df["postcode_district"] = df["postcode"].str.split().str[0]

df["postcode_district"]

0          E3
1         E17
2         NW4
3          E2
4         E14
         ... 
72587     E10
72588    RM13
72589     RM3
72590     E18
72591    RM13
Name: postcode_district, Length: 72592, dtype: object

In [46]:
# Property Type: Flat, Terraced, Semi-Detached or Detached
type_order = {"F": 1, "T": 2, "S": 3, "D": 4}
df["property_type_ordinal"] = df["property_type"].map(type_order).astype("int64")

df["property_type_ordinal"]

0        1
1        1
2        3
3        1
4        1
        ..
72587    1
72588    3
72589    2
72590    3
72591    2
Name: property_type_ordinal, Length: 72592, dtype: int64

In [47]:
# Flat or not
df["is_flat"]  = (df["property_type"] == "F").astype("int64")

df["is_flat"]

0        1
1        1
2        0
3        1
4        1
        ..
72587    1
72588    0
72589    0
72590    0
72591    0
Name: is_flat, Length: 72592, dtype: int64

In [48]:
# Sales count in District: How many properties are sold in the district
district_counts = df["postcode_district"].value_counts()
df["district_sales_count"] = df["postcode_district"].map(district_counts).astype("int64")

df["district_sales_count"]

0         530
1        1046
2         212
3         330
4         879
         ... 
72587     450
72588     314
72589     402
72590     226
72591     314
Name: district_sales_count, Length: 72592, dtype: int64

In [53]:
# Log Price
df["log_price"] = np.log10(df["price"])

df["log_price"]

0        5.332438
1        5.542825
2        5.732394
3        5.662758
4        5.423246
           ...   
72587    5.595276
72588    5.588832
72589    5.553883
72590    6.039414
72591    5.526339
Name: log_price, Length: 72592, dtype: float64

In [50]:
print(f"Unique Postcode Areas: {df['postcode_area'].nunique()}")
print(f"Unique Postcode Districts: {df['postcode_district'].nunique()}")
print(f"District Sales Count Range: {df['district_sales_count'].min()} to {df['district_sales_count'].max()}")
print(f"Property type ordinal counts:\n{df['property_type_ordinal'].value_counts().sort_index()}")
print(f"Flat distribution: {df['is_flat'].mean():.1%} flats")
print(f"Log price skewness: {df['log_price'].skew():.2f}")

Unique Postcode Areas: 21
Unique Postcode Districts: 273
District Sales Count Range: 1 to 1339
Property type ordinal counts:
property_type_ordinal
1    36826
2    20901
3    11427
4     3438
Name: count, dtype: int64
Flat distribution: 50.7% flats
Log price skewness: 0.69


In [54]:
df

,price,date_of_transfer,postcode,property_type,old_new,duration,district,postcode_area,postcode_district,property_type_ordinal,district_sales_count,log_price,is_flat
0,215000,2025-01-31,E3 2PQ,F,N,L,TOWER HAMLETS,E,E3,1,530,5.332438,1
1,349000,2025-12-22,E17 7LB,F,N,L,WALTHAM FOREST,E,E17,1,1046,5.542825,1
2,540000,2025-12-16,NW4 3PG,S,N,F,BARNET,NW,NW4,3,212,5.732394,0
3,460000,2025-12-15,E2 8FZ,F,N,L,HACKNEY,E,E2,1,330,5.662758,1
4,265000,2025-12-19,E14 9BF,F,N,L,TOWER HAMLETS,E,E14,1,879,5.423246,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
72587,393800,2025-06-11,E10 7PE,F,N,L,WALTHAM FOREST,E,E10,1,450,5.595276,1
72588,388000,2025-05-29,RM13 9SN,S,N,F,HAVERING,RM,RM13,3,314,5.588832,0
72589,358000,2025-06-05,RM3 7DX,T,N,F,HAVERING,RM,RM3,2,402,5.553883,0
72590,1095000,2025-01-03,E18 2PS,S,N,F,REDBRIDGE,E,E18,3,226,6.039414,0
